# 手机价格分类

In [168]:
import torch
from torch.utils.data import TensorDataset # 数据集对象
from torch.utils.data import DataLoader    # 数据加载器
import torch.nn as nn
import torch.optim as optim                # 优化器

from sklearn.model_selection import train_test_split    # 切分训练集和测试集
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

from torchsummary import summary
from torchvision.transforms.v2.functional import affine


## 构建数据集

In [169]:
def create_dataset():
    # 加载csv文件数据
    data = pd.read_csv('./data/手机价格预测.csv')
    # print(f'data:{data.head()}')
    # print(f'data.shape:{data.shape}')

    # 获取x特征列和y标签列
    x, y = data.iloc[:, :-1], data.iloc[:, -1]
    x = x.astype(np.float32)

    # 切分训练集和测试集
    # 特征，标签，测试集比例，随机种子，样本的分布(参考y的类别进行抽取数据，保证y的不同类别在测试集和训练集中比例相同)
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=24, stratify= y)

    # 封装数据集
    dataset_train = TensorDataset(torch.tensor(x_train.values), torch.tensor(y_train.values))
    dataset_test = TensorDataset(torch.tensor(x_test.values), torch.tensor(y_test.values))

    # 返回结果                              输入特征数           输出标签数(unique能提取不重复的数)
    return dataset_train, dataset_test, x_train.shape[1], len(y_train.unique())

## 搭建神经网络

In [170]:
class PhonePriceModel(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.linear1 = nn.Linear(input_size, 64)
        self.linear2 = nn.Linear(64, 256)
        self.linear3 = nn.Linear(256, 128)
        self.linear4 = nn.Linear(128, 32)
        self.output = nn.Linear(32, output_size)
        self.bn1 = nn.BatchNorm1d(num_features = 64, eps = 1e-05, momentum = 0.1, affine = True, track_running_stats = True)

    def forward(self, x):
        x = self.bn1(self.linear1(x))
        x = torch.relu(x)
        x = torch.relu(self.linear2(x))
        x = torch.relu(self.linear3(x))
        x = torch.relu(self.linear4(x))
        x = self.output(x)
        return x

## 编写训练函数

In [171]:
def train(train_dataset, input_size, output_size):
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    model = PhonePriceModel(input_size, output_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    epochs = 100
    for epoch in range(epochs):
        total_loss, batch_num = 0.0, 0
        start = time.time()
        for x, y in train_loader:
            # 切换模式
            model.train()

            # 计算损失
            y_pred = model(x)
            loss = criterion(y_pred, y)

            # 梯度清零，反向传播，优化参数
            optimizer.zero_grad()
            loss.sum().backward()
            optimizer.step()
            total_loss += loss.sum().item()
            batch_num += 1

       # 至此，本轮训练结束，打印训练结果
        print(f'epoch:{epoch+1}/{epochs}  loss:{total_loss/batch_num:.4f}  time:{time.time()-start:.2f}s')

    # 全部训练结束，保存模型(参数)
    # 参1：模型对象参数(权重，偏置矩阵)  参2：模型保存的文件名
    # print(f'\n\n模型参数信息：{model.state_dict()}\n\n')
    torch.save(model.state_dict(), './model/phone_price_model.pth') # 后缀名用pth, pkl, pickle均可


## 模型测试

In [172]:
def evaluate(test_dataset, input_size, output_size):
    # 创建神经网络对象
    model = PhonePriceModel(input_size, output_size)
    model.load_state_dict(torch.load('./model/phone_price_model.pth'))

    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    # 定义变量：记录预测正确的样本数
    correct = 0
    for x, y in test_loader:
        # 模型切换为测试模式
        model.eval()
        y_pred = model(x)
        # 根据加权求和得到类别，用argmax()获取最大值对应下标，就是类别
        y_pred = torch.argmax(y_pred, dim=1)    # dim=1表示按行处理求最大值对应的下标
        # print(f'y_pred:{y_pred}')
        # print(f'y:{y}')
        correct += (y_pred == y).sum().item()
    print(f'正确率：{correct/len(test_dataset):.2%}')


In [173]:
if __name__ == '__main__':
    train_dataset, test_dataset, input_size, output_size = create_dataset()
    model = PhonePriceModel(input_size, output_size)
    # summary(model, input_size = (16, input_size), device = 'cpu') # 每批16条，每条input_size特征
    train(train_dataset, input_size, output_size)
    evaluate(test_dataset, input_size, output_size)

epoch:1/100  loss:1.3160  time:0.25s
epoch:2/100  loss:1.0303  time:0.27s
epoch:3/100  loss:0.6854  time:0.25s
epoch:4/100  loss:0.6037  time:0.34s
epoch:5/100  loss:0.5460  time:0.25s
epoch:6/100  loss:0.5303  time:0.26s
epoch:7/100  loss:0.4838  time:0.24s
epoch:8/100  loss:0.4882  time:0.26s
epoch:9/100  loss:0.5072  time:0.23s
epoch:10/100  loss:0.4670  time:0.23s
epoch:11/100  loss:0.4868  time:0.22s
epoch:12/100  loss:0.4974  time:0.22s
epoch:13/100  loss:0.4578  time:0.22s
epoch:14/100  loss:0.4831  time:0.21s
epoch:15/100  loss:0.4599  time:0.21s
epoch:16/100  loss:0.5076  time:0.21s
epoch:17/100  loss:0.4539  time:0.21s
epoch:18/100  loss:0.4462  time:0.21s
epoch:19/100  loss:0.5295  time:0.21s
epoch:20/100  loss:0.4200  time:0.20s
epoch:21/100  loss:0.5156  time:0.21s
epoch:22/100  loss:0.4543  time:0.20s
epoch:23/100  loss:0.4319  time:0.21s
epoch:24/100  loss:0.4561  time:0.21s
epoch:25/100  loss:0.4655  time:0.21s
epoch:26/100  loss:0.4309  time:0.21s
epoch:27/100  loss:0.